# Ablation du transfer learning — segmentation des plans d'eau

Question : **le pré-entraînement ImageNet aide-t-il vraiment ?** On entraîne le même U-Net ResNet34 avec la
même recette (mêmes données, mêmes augmentations, même perte, même graine, mêmes epochs) en ne changeant
que l'initialisation de l'encodeur :

| Variante | Encodeur | Ce qu'on entraîne |
|---|---|---|
| `scratch` | aléatoire | tout |
| `frozen` | ImageNet | décodeur seulement (encodeur gelé, BatchNorm figé) |
| `finetune` | ImageNet | tout, même learning rate |
| `finetune_discriminative` | ImageNet | tout, encodeur à LR x0.1 (le décodeur apprend plus vite) |

Plus une courbe d'**efficacité des données** : `scratch` et `finetune` entraînés sur 10 % et 25 % des données
(le transfer learning compte surtout quand les données sont rares).

1. **Add Input** : dataset `franciscoescobar/satellite-images-of-water-bodies` (à refaire après un import `.ipynb`).
2. Accelerator = **GPU**, internet **On** (téléchargement des poids ImageNet).
3. **Save Version > Save & Run All (Commit)** : ~1 h sur T4. Les résultats s'écrivent run par run, un crash n'efface pas les précédents.
4. Télécharge depuis l'Output :
   - `variants/model_*.pt` -> `models/variants/`
   - `ablation_metrics.json` -> `reports/ablation_kaggle.json`
5. En local : `python -m src.compare_variants` ajoute le score de chaque variante sur les vrais barrages tunisiens.

In [ ]:
!pip install -q segmentation-models-pytorch mlflow

In [ ]:
import json
import os
import random

import numpy as np
import segmentation_models_pytorch as smp
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_ROOT = '/kaggle/input'


def print_tree(root, max_depth=4):
    for dirpath, dirnames, files in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth >= max_depth:
            dirnames[:] = []
        print('  ' * depth + (os.path.basename(dirpath) or dirpath) + '/', f'({len(files)} fichiers)' if files else '')


def find_dir(root, name_options):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() in name_options:
                return os.path.join(dirpath, d)
    return None


if not os.path.isdir(INPUT_ROOT) or not os.listdir(INPUT_ROOT):
    raise FileNotFoundError(
        'Aucun dataset attaché (' + INPUT_ROOT + ' est vide). Dans le panneau de droite : '
        '"Add Input" > cherche "satellite-images-of-water-bodies" (franciscoescobar) > Add, '
        'puis relance toutes les cellules.'
    )

DATA_ROOT = find_dir(INPUT_ROOT, {'images', 'image'})
MASK_ROOT = find_dir(INPUT_ROOT, {'masks', 'mask'})
if DATA_ROOT is None or MASK_ROOT is None:
    print('Contenu de', INPUT_ROOT, ':')
    print_tree(INPUT_ROOT)
    raise FileNotFoundError(
        'Dossiers Images/Masks introuvables sous ' + INPUT_ROOT + ' (arborescence ci-dessus). '
        'Vérifie que le dataset attaché est bien "satellite-images-of-water-bodies", '
        'ou fixe DATA_ROOT / MASK_ROOT à la main.'
    )

# Pair images and masks by name without extension, so a .jpg image still
# finds its .png mask, and any image without a mask is skipped (and counted).
masks_by_stem = {os.path.splitext(f)[0]: f for f in os.listdir(MASK_ROOT)}
PAIRS = sorted(
    (f, masks_by_stem[os.path.splitext(f)[0]])
    for f in os.listdir(DATA_ROOT)
    if os.path.splitext(f)[0] in masks_by_stem
)
n_images = len(os.listdir(DATA_ROOT))
print('Images:', DATA_ROOT, '->', n_images, 'fichiers')
print('Masks:', MASK_ROOT, '->', len(masks_by_stem), 'fichiers')
print('Paires image/masque:', len(PAIRS), '| images sans masque ignorées:', n_images - len(PAIRS))
if not PAIRS:
    raise FileNotFoundError('Aucune image n\'a de masque du même nom dans ' + MASK_ROOT)

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 16

# Same plain resize as the API (src/serve.py) — no letterbox: the model
# must see at serving time exactly what it saw in training.
resize_image = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
resize_mask = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


class WaterBodiesDataset(Dataset):
    """With augment=True: 30% of the time a black no-data strip labelled
    land (v1 learned "dark = water" and flagged tile-edge no-data as water),
    then random flips and 90-degree rotations (orientation is arbitrary in
    satellite imagery, so these are free extra training data)."""

    def __init__(self, pairs, augment):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_name, mask_name = self.pairs[idx]
        image = transforms.functional.to_tensor(resize_image(Image.open(os.path.join(DATA_ROOT, image_name)).convert('RGB')))
        mask = (transforms.functional.to_tensor(resize_mask(Image.open(os.path.join(MASK_ROOT, mask_name)).convert('L'))) > 0.5).float()

        if self.augment:
            if random.random() < 0.3:
                width = random.randint(IMAGE_SIZE // 10, IMAGE_SIZE // 3)
                image[:, -width:, :] = 0
                mask[:, -width:, :] = 0
            if random.random() < 0.5:
                image, mask = image.flip(-1), mask.flip(-1)
            if random.random() < 0.5:
                image, mask = image.flip(-2), mask.flip(-2)
            k = random.randint(0, 3)
            image, mask = torch.rot90(image, k, (-2, -1)), torch.rot90(mask, k, (-2, -1))

        return normalize(image), mask


# Fixed-seed split, so the test set is the same on every run and never
# used for any decision (model selection uses val only).
perm = torch.randperm(len(PAIRS), generator=torch.Generator().manual_seed(SEED)).tolist()
n_holdout = int(0.15 * len(PAIRS))
test_files = [PAIRS[i] for i in perm[:n_holdout]]
val_files = [PAIRS[i] for i in perm[n_holdout:2 * n_holdout]]
train_files = [PAIRS[i] for i in perm[2 * n_holdout:]]

train_loader = DataLoader(WaterBodiesDataset(train_files, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(WaterBodiesDataset(val_files, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(WaterBodiesDataset(test_files, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('train:', len(train_files), 'val:', len(val_files), 'test:', len(test_files))

In [ ]:
import time

import matplotlib.pyplot as plt
import mlflow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 20
LR = 3e-4
THRESHOLDS = [round(float(t), 2) for t in np.arange(0.3, 0.81, 0.05)]
VARIANT_DIR = '/kaggle/working/variants'
os.makedirs(VARIANT_DIR, exist_ok=True)

VARIANTS = {
    'scratch': {'weights': None, 'frozen': False, 'encoder_lr_mult': 1.0},
    'frozen': {'weights': 'imagenet', 'frozen': True, 'encoder_lr_mult': 1.0},
    'finetune': {'weights': 'imagenet', 'frozen': False, 'encoder_lr_mult': 1.0},
    'finetune_discriminative': {'weights': 'imagenet', 'frozen': False, 'encoder_lr_mult': 0.1},
}
RUNS = [(name, 1.0) for name in VARIANTS] + [(name, f) for name in ('scratch', 'finetune') for f in (0.1, 0.25)]

dice = smp.losses.DiceLoss(mode='binary')
bce = smp.losses.SoftBCEWithLogitsLoss()


def counts(model, loader, thresholds):
    model.eval()
    result = {t: [0, 0, 0] for t in thresholds}
    with torch.no_grad():
        for images, masks in loader:
            probs = torch.sigmoid(model(images.to(device)))
            masks = masks.to(device).bool()
            for t in thresholds:
                pred = probs > t
                result[t][0] += (pred & masks).sum().item()
                result[t][1] += (pred & ~masks).sum().item()
                result[t][2] += (~pred & masks).sum().item()
    return result


def iou(tp, fp, fn):
    return tp / max(tp + fp + fn, 1)


mlflow.set_tracking_uri('sqlite:////kaggle/working/mlflow.db')
mlflow.set_experiment('transfer-learning-ablation')


def train_run(variant, fraction):
    config = VARIANTS[variant]
    torch.manual_seed(SEED)
    random.seed(SEED)

    subset = train_files[:max(int(fraction * len(train_files)), BATCH_SIZE)]
    loader = DataLoader(WaterBodiesDataset(subset, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    model = smp.Unet(encoder_name='resnet34', encoder_weights=config['weights'], in_channels=3, classes=1).to(device)
    if config['frozen']:
        for p in model.encoder.parameters():
            p.requires_grad = False
        groups = [{'params': list(model.decoder.parameters()) + list(model.segmentation_head.parameters()), 'lr': LR}]
    else:
        groups = [
            {'params': model.encoder.parameters(), 'lr': LR * config['encoder_lr_mult']},
            {'params': list(model.decoder.parameters()) + list(model.segmentation_head.parameters()), 'lr': LR},
        ]
    optimizer = torch.optim.Adam(groups)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    curve, best_iou, best_epoch, best_state = [], -1.0, 0, None
    started = time.time()
    with mlflow.start_run(run_name=f'{variant}@{fraction}'):
        mlflow.log_params({'variant': variant, 'data_fraction': fraction, 'n_train': len(subset), 'epochs': EPOCHS,
                           'lr': LR, 'encoder_lr_mult': config['encoder_lr_mult'], 'frozen': config['frozen'],
                           'weights': str(config['weights']), 'seed': SEED})
        for epoch in range(EPOCHS):
            model.train()
            if config['frozen']:
                model.encoder.eval()  # frozen means BatchNorm statistics too
            for images, masks in loader:
                images, masks = images.to(device), masks.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                (dice(outputs, masks) + bce(outputs, masks)).backward()
                optimizer.step()
            scheduler.step()

            val_iou = iou(*counts(model, val_loader, [0.5])[0.5])
            curve.append(round(val_iou, 4))
            mlflow.log_metric('val_iou', val_iou, step=epoch)
            if val_iou > best_iou:
                best_iou, best_epoch = val_iou, epoch + 1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'  {variant}@{fraction} epoch {epoch + 1}/{EPOCHS} val_iou={val_iou:.4f} best={best_iou:.4f}@{best_epoch}')

        model.load_state_dict(best_state)
        val_counts = counts(model, val_loader, THRESHOLDS)
        threshold = max(THRESHOLDS, key=lambda t: iou(*val_counts[t]))
        tp, fp, fn = counts(model, test_loader, [threshold])[threshold]
        result = {
            'variant': variant, 'data_fraction': fraction, 'n_train': len(subset),
            'best_epoch': best_epoch, 'threshold': threshold,
            'val_iou': round(iou(*val_counts[threshold]), 4), 'test_iou': round(iou(tp, fp, fn), 4),
            'test_precision': round(tp / max(tp + fp, 1), 4), 'test_recall': round(tp / max(tp + fn, 1), 4),
            'val_iou_curve': curve, 'minutes': round((time.time() - started) / 60, 1),
        }
        mlflow.log_metrics({'test_iou': result['test_iou'], 'best_epoch': best_epoch})

    if fraction == 1.0:
        # Same checkpoint format as the production model, so src/evaluate.py can score it as-is.
        torch.save({'state_dict': model.state_dict(), 'encoder_name': 'resnet34', 'image_size': IMAGE_SIZE,
                    'threshold': threshold, 'metrics': {k: v for k, v in result.items() if k != 'val_iou_curve'}},
                   f'{VARIANT_DIR}/model_{variant}.pt')
    return result

In [ ]:
results = {}
for variant, fraction in RUNS:
    key = f'{variant}@{fraction}'
    print('=====', key)
    results[key] = train_run(variant, fraction)
    print(json.dumps({k: v for k, v in results[key].items() if k != 'val_iou_curve'}))
    with open('/kaggle/working/ablation_metrics.json', 'w') as f:
        json.dump(results, f, indent=2)
print('done:', list(results))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for variant in VARIANTS:
    axes[0].plot(range(1, EPOCHS + 1), results[f'{variant}@1.0']['val_iou_curve'], label=variant)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('validation IoU'); axes[0].set_title('Convergence (100% of the data)')
axes[0].legend(); axes[0].grid(alpha=0.3)

for variant in ('scratch', 'finetune'):
    fractions = sorted({r['data_fraction'] for k, r in results.items() if r['variant'] == variant})
    axes[1].plot([f * 100 for f in fractions],
                 [results[f'{variant}@{f}']['test_iou'] for f in fractions], marker='o', label=variant)
axes[1].set_xlabel('% of the training data'); axes[1].set_ylabel('test IoU'); axes[1].set_title('Data efficiency')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/ablation_curves.png')
plt.show()

print(f"{'run':32s} {'best ep':>8s} {'thr':>5s} {'val IoU':>8s} {'test IoU':>9s} {'prec':>6s} {'recall':>7s}")
for key, r in results.items():
    print(f"{key:32s} {r['best_epoch']:8d} {r['threshold']:5.2f} {r['val_iou']:8.4f} {r['test_iou']:9.4f} {r['test_precision']:6.3f} {r['test_recall']:7.3f}")